# Procedimento para calcular população de bacias hidrossanitárias
##### Este documento define as etapas para a obtenção do número de habitantes inseridos na Área de Prestação de Serviços de bacias de esgotamento

Importação de bibliotecas:

In [145]:
import pandas as pd
import geopandas as gpd
import os

## 1º Passo: Importação dos dados
##### Setores Censitários: https://www.ibge.gov.br/estatisticas/sociais/trabalho/22827-censo-demografico-2022.html?edicao=41852&t=resultados 
Fazer download da malha de setores centiários por UF
##### Domicílios: https://www.ibge.gov.br/estatisticas/sociais/populacao/38734-cadastro-nacional-de-enderecos-para-fins-estatisticos.html?edicao=38891&t=resultados
Selecionar arquivos por município
##### APS encaminhado pela CORSAN; Bacias delimitadas pelo Analista. Coluna com o nome das bacias também deve ser especificado.

*Buscar código do município em https://www.ibge.gov.br/explica/codigos-dos-municipios.php

In [146]:
#Caçapava do Sul
setores = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Caçapava do Sul\Arquivos Baixados\RS_setores_CD2022.gpkg')
domicilios = pd.read_csv(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Caçapava do Sul\Arquivos Baixados\4302808\4302808.csv',delimiter = ';')
aps = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Caçapava do Sul\Arquivos Criados\APS.gpkg')
bacias = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Caçapava do Sul\Arquivos Baixados\Shapefiles Tomo I\SB_CS_ok.shp')
coluna_nome_bacias = 'Nome'
caminho_exportacao = r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Caçapava do Sul\popdom_bacia_2022.xlsx'
crs = "EPSG:31982"

## Funções auxiliares

In [147]:
#Funções auxiliares

#contagem de domicílios em cada setor na APS

import geopandas as gpd

def somar_extensao_polig(lines_gdf, polys_gdf, poly_id_col="Nome"):
    """
    Retorna um DataFrame com o comprimento (m e km) das LINHAS contidas em cada polígono.
    Requer ambos em CRS projetado (unidades em METROS).
    - Filtra geometrias nulas/vazias
    - Valida polígonos (make_valid/buffer(0))
    - Faz overlay com keep_geom_type=False e filtra só linhas
    - Soma por polígono e adiciona coluna em km
    """
    # Cópias e colunas necessárias
    lines = lines_gdf[["geometry"]].copy()
    polys = polys_gdf[[poly_id_col, "geometry"]].copy()

    # CRS: alinhar se necessário
    if lines.crs != polys.crs:
        polys = polys.to_crs(lines.crs)

    # Limpeza: remover nulos/vazios
    lines = lines[lines.geometry.notnull() & ~lines.geometry.is_empty]
    polys = polys[polys.geometry.notnull() & ~polys.geometry.is_empty]

    # Validar polígonos (Shapely 2 -> make_valid; fallback buffer(0))
    try:
        polys["geometry"] = polys.geometry.make_valid()
    except Exception:
        polys["geometry"] = polys.buffer(0)

    # Overlay SEM restringir tipo de geometria
    inter = gpd.overlay(lines, polys, how="intersection", keep_geom_type=False)

    # Ficar só com partes lineares (descarta GeometryCollection/Polígonos/Pontos)
    inter = inter[inter.geom_type.isin(["LineString", "MultiLineString"])].copy()

    if inter.empty:
        # Retorno “vazio” com colunas esperadas
        return gpd.GeoDataFrame(
            {poly_id_col: [], "Extensão de Rede (m)": [], "Extensão de Rede (km)": []}
        )

    # Comprimento em metros (CRS deve estar em metros!)
    inter["Extensão de Rede (m)"] = inter.geometry.length

    # Soma por polígono
    out = (
        inter.groupby(poly_id_col, as_index=False)["Extensão de Rede (m)"]
        .sum()
        .sort_values(poly_id_col)
    )
    out["Extensão de Rede (km)"] = out["Extensão de Rede (m)"] / 1000.0
    return out

    return out

def contar_pontos_poligono(polygons, points, polygon_id_col="poly_id", predicate="intersects"):
    if polygons.crs != points.crs:
        points = points.to_crs(polygons.crs)

    # Garante coluna de ID
    if polygon_id_col not in polygons.columns:
        polygons = polygons.reset_index(drop=False).rename(columns={"index": polygon_id_col})

    joined = gpd.sjoin(points, polygons[[polygon_id_col, "geometry"]], predicate=predicate)
    counts = joined.groupby(polygon_id_col).size().rename("n_pontos").reset_index()
    
    result = polygons.merge(counts, on=polygon_id_col, how="left")
    result["n_pontos"] = result["n_pontos"].fillna(0).astype(int)
    
    return result

# Não é necessário mexer nisso abaixo

## 2º Passo: Tratamento dos dados

Conforme Diretriz Corsan (2025), o IBGE considera, para a densidade domiciliar, somente os domicílios particulares ocupados (v0007), o qual não representa a realidade das economias residenciais no cadastro da Corsan/Aegea. Portanto, deve-se recalcular a densidade domiciliar dos setores censitários. Para recalcular a densidade domiciliar, deve-se dividir a população (v0001) pelo total de domicílios particulares (v0003), gerando uma nova coluna “Densidade”.

In [148]:
setores['Densidade'] = setores['v0001']/setores['v0003']

É necessário filtrar os domicílios particulares (COD_ESPECIE = 1) e igrejas (COD_ESPECIE = 8) e transformar csv de domicílios em um arquivo georreferenciado

In [149]:
domparticular = domicilios[(domicilios['COD_ESPECIE'] == 1) |(domicilios['COD_ESPECIE'] == 8) ]
domparticular = gpd.GeoDataFrame(domparticular, geometry=gpd.points_from_xy(domparticular.LONGITUDE, domparticular.LATITUDE), crs="EPSG:4326")

Deve-se colocar tudo no mesmo CRS definido

In [150]:
domparticular = domparticular.to_crs(crs)
bacias = bacias.to_crs(crs)
aps = aps.to_crs(crs)
setores = setores.to_crs(crs) #convertendo pra sistema de coordenadas padrão

##### Intersecção entre setores e APS

In [151]:
setores_aps = gpd.clip(setores, aps) #interseção entre setores e APS
domparticular_aps = gpd.clip(domparticular, aps) #interseção entre domicílios e APS

## 3º Passo: Cálculo da população na APS
##### As etapas realizadas são:
- Intersecção entre Setores e APS
- Intersecção entre Domicílios e APS
- Contagem de domicílios em cada setor na APS
- Calculo da população

##### Contagem de domicílios em cada setor na APS

In [152]:
populacao_aps = contar_pontos_poligono(setores_aps, domparticular_aps)

##### Cálculo da população com a densidade e n_pontos criado

In [153]:
populacao_aps['População 2022'] = populacao_aps['Densidade']*populacao_aps['n_pontos']

pop = populacao_aps['População 2022'].sum()
print(f"A população total na APS em 2022 é de {pop:.0f}")
econ = len(domparticular_aps)
print(f"A população total na APS em 2022 é de {econ:.0f}")

A população total na APS em 2022 é de 26457
A população total na APS em 2022 é de 13439


## 4º Passo: Cálculo da população por bacia (2022)

Após delimitar as bacias para pelo menos 90% dos domicílios do IBGE (Censo 2022), deverá ser identificado quantos domicílios estão inseridos em cada bacia. As etapas realizadas são:
- Criação de camada com domicílios classificados por setor e bacia
- Criação de camada com bacias divididas em setores
- Calculo da população com base na densidade de cada domicílio dentro de cada setor dividido pela bacia
- Agrupamento dos valores por bacia, gerando a quantidade de população e domicílios por bacia

##### Intersecções entre domicílios, setores e bacias

In [154]:
camada_unida = gpd.sjoin(
    domparticular_aps,
    bacias, 
    predicate="intersects",
    how="left"
)
camada_unida = camada_unida.drop(columns=['index_right'], errors='ignore')

dompart_setores = gpd.sjoin(
    camada_unida,
    populacao_aps,  
    predicate="intersects",
    how="left"
)

bacias_setores = gpd.sjoin(
    populacao_aps,
    bacias,  
    predicate="intersects",
    how="left"
)

dompart_setores_filtrado = dompart_setores[[coluna_nome_bacias, 'CD_SETOR']]
bacias_setores_filtrado = bacias_setores[[coluna_nome_bacias, 'CD_SETOR','Densidade']]

##### Contagem da quantidade de vezes que uma combinação Setores Censitários + Bacia aparece

In [155]:
# Passo 1: Contar ocorrências de Nome + CD_SETOR na planilha de referência
contagem = (
    dompart_setores_filtrado
    .groupby([coluna_nome_bacias, 'CD_SETOR'])
    .size()
    .reset_index(name='Domicílios')
)

# Passo 2: Fazer merge com o DataFrame base
bacias_setores_filtrado = bacias_setores_filtrado.merge(contagem, on=[coluna_nome_bacias, 'CD_SETOR'], how='left')

# Passo 3: Substituir NaN por 0 (caso não tenha ocorrência)
bacias_setores_filtrado['Domicílios'] = bacias_setores_filtrado['Domicílios'].fillna(0).astype(int)

##### Cálculo da população por combinação Setores Censitários + Bacia

In [156]:
bacias_setores_filtrado['População'] = bacias_setores_filtrado['Domicílios']*bacias_setores_filtrado['Densidade']

##### Soma da população calculada por bacia

In [157]:
bacias_populacao = bacias_setores_filtrado[[coluna_nome_bacias,'Domicílios','População']].groupby(coluna_nome_bacias).sum()
bacias_populacao

,Domicílios,População
Nome,,
AR1,411,906.763113
AR2,120,237.822137
AR3,346,711.899952
BT1,230,469.234716
CE1,1293,2615.269165
CE2,379,774.316691
CE3,652,1267.900622
CE4,1441,2831.571343
CE5,2758,5050.873211


##### Exportar excel final

In [158]:
bacias_populacao.to_excel(caminho_exportacao)

# Resultados

In [159]:
pop_aps = populacao_aps['População 2022'].sum()
print(f"A população total na APS em 2022 é de {pop_aps:.0f}")
dom_aps = len(domparticular_aps)
print(f"Os domicílios totais na APS em 2022 é de {dom_aps:.0f}")

resultado_dompop = bacias_populacao.copy()

resultado_dompop['Dom % bacias'] = resultado_dompop['Domicílios']/(resultado_dompop['Domicílios'].sum())
resultado_dompop['Dom % APS'] = resultado_dompop['Domicílios']/dom_aps
resultado_dompop['Pop % bacias'] = resultado_dompop['População']/(resultado_dompop['População'].sum())
resultado_dompop['Pop % APS'] = resultado_dompop['População']/pop_aps

display(resultado_dompop)

A população total na APS em 2022 é de 26457
Os domicílios totais na APS em 2022 é de 13439


,Domicílios,População,Dom % bacias,Dom % APS,Pop % bacias,Pop % APS
Nome,,,,,,
AR1,411,906.763113,0.032841,0.030583,0.035824,0.034273
AR2,120,237.822137,0.009588,0.008929,0.009396,0.008989
AR3,346,711.899952,0.027647,0.025746,0.028125,0.026907
BT1,230,469.234716,0.018378,0.017114,0.018538,0.017736
CE1,1293,2615.269165,0.103316,0.096213,0.103322,0.098849
CE2,379,774.316691,0.030284,0.028202,0.030591,0.029267
CE3,652,1267.900622,0.052097,0.048516,0.050091,0.047922
CE4,1441,2831.571343,0.115142,0.107225,0.111868,0.107024
CE5,2758,5050.873211,0.220376,0.205224,0.199547,0.190906


# Extensão e Área das Bacias

In [160]:
eixolog = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Caçapava do Sul\copia eixolog aumentado.gpkg')
eixolog =eixolog.to_crs(crs)
bacias_area = bacias.to_crs(crs)

bacias_area_len = somar_extensao_polig(eixolog, bacias_area, poly_id_col="Nome")
bacias_area_len["Área (km²)"] = bacias_area.geometry.area / 10**6

bacias_area_len = bacias_area_len.set_index("Nome")

resultado_final = resultado_dompop.merge(
    bacias_area_len,
    left_index=True,
    right_index=True,
    how="left"
)

resultado_final = resultado_final[['Domicílios','Dom % bacias','Dom % APS','População','Pop % bacias','Pop % APS','Extensão de Rede (m)','Área (km²)']].transpose()

display(resultado_final)

Nome,AR1,AR2,AR3,BT1,CE1,CE2,CE3,CE4,CE5,CJ1,...,NS1,NS2,NS3,SR1,SR2,SR3,VB1,VS1,VS2,VS3
Domicílios,411.000000,120.000000,346.000000,230.000000,1293.000000,379.000000,652.000000,1441.000000,2758.000000,830.000000,...,43.000000,653.000000,47.000000,325.000000,383.000000,194.000000,103.000000,62.000000,1232.000000,149.000000
Dom % bacias,0.032841,0.009588,0.027647,0.018378,0.103316,0.030284,0.052097,0.115142,0.220376,0.066320,...,0.003436,0.052177,0.003755,0.025969,0.030603,0.015501,0.008230,0.004954,0.098442,0.011906
Dom % APS,0.030583,0.008929,0.025746,0.017114,0.096213,0.028202,0.048516,0.107225,0.205224,0.061761,...,0.003200,0.048590,0.003497,0.024183,0.028499,0.014436,0.007664,0.004613,0.091673,0.011087
População,906.763113,237.822137,711.899952,469.234716,2615.269165,774.316691,1267.900622,2831.571343,5050.873211,1833.439148,...,85.651822,1445.290121,72.178571,645.455051,755.545887,389.119447,180.611123,106.731759,2746.677512,320.318898
Pop % bacias,0.035824,0.009396,0.028125,0.018538,0.103322,0.030591,0.050091,0.111868,0.199547,0.072434,...,0.003384,0.057100,0.002852,0.025500,0.029850,0.015373,0.007135,0.004217,0.108514,0.012655
Pop % APS,0.034273,0.008989,0.026907,0.017736,0.098849,0.029267,0.047922,0.107024,0.190906,0.069298,...,0.003237,0.054627,0.002728,0.024396,0.028557,0.014707,0.006827,0.004034,0.103815,0.012107
Extensão de Rede (m),7551.806345,1793.815786,3885.147136,2522.505613,12871.652970,3698.692654,6643.140839,11098.799583,22515.468484,7660.857575,...,520.054627,7906.584254,1867.258236,3209.154388,3686.180513,1354.706096,3857.053111,2472.206887,11703.603165,1066.995710
Área (km²),0.338039,0.947921,1.010066,0.691316,0.839470,0.224701,1.562092,0.769587,0.555546,0.600816,...,0.707369,1.830807,0.333413,0.643375,0.319291,1.148275,0.287620,0.261939,0.539322,2.267581


In [162]:
resultado_final.to_excel(os.path.join(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Caçapava do Sul\inputpredim.xlsx'))